# Per-entity metrics — train / validation / test

Charge le modele entraine dans `outputs/bio/bert/baseline/final_model/` et calcule les metriques **par entite** (Precision / Recall / F1 / Support) sur les 3 splits.

Autonome : aucune dependance a `src/`. Requiert `torch`, `transformers`, `datasets`, `seqeval`, `pandas`.

Lance tout : **Kernel -> Restart & Run All**. (Le split `train` = 153k exemples, c'est le plus lent, surtout sur CPU.)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import classification_report

# Racine = le dossier qui contient bio_tagging/ et outputs/
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "bio_tagging").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

MODEL_DIR = REPO_ROOT / "outputs" / "bio" / "bert" / "baseline" / "final_model"
DATA_DIR = REPO_ROOT / "bio_tagging" / "bio_dataset" / "bert-base-cased"

SPLITS = ["validation", "test", "train"]  # enleve "train" s'il est trop lent
BATCH_SIZE = 64

# mps = GPU Apple Silicon (rapide) ; sinon cuda ; sinon cpu.
# Si une erreur MPS apparait, force device = "cpu".
if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Repo root :", REPO_ROOT)
print("Modele    :", MODEL_DIR, "| existe:", MODEL_DIR.exists())
print("Dataset   :", DATA_DIR, "| existe:", DATA_DIR.exists())
print("Device    :", device)

Repo root : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/deb_bert_version/deb_bert_version_reg
Modele    : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/deb_bert_version/deb_bert_version_reg/outputs/bio/bert/baseline/final_model | existe: False
Dataset   : /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/deb_bert_version/deb_bert_version_reg/bio_tagging/bio_dataset/bert-base-cased | existe: True
Device    : mps


In [2]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR)).eval().to(device)
id2label = {int(k): v for k, v in model.config.id2label.items()}

dataset = load_from_disk(str(DATA_DIR))
collator = DataCollatorForTokenClassification(tokenizer)

print(len(id2label), "labels |", {k: len(v) for k, v in dataset.items()})

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/deb_bert_version/deb_bert_version_reg/outputs/bio/bert/baseline/final_model'. Use `repo_type` argument if needed.

### Fonctions : inference + metriques par entite

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x


def predict_split(split):
    """Inference sur un split -> (true_labels, true_preds) en sequences de tags."""
    loader = DataLoader(dataset[split], batch_size=BATCH_SIZE, collate_fn=collator)
    true_labels, true_preds = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=split):
            labels = batch.pop("labels").numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(-1).cpu().numpy()
            for pred_row, label_row in zip(preds, labels):
                seq_true, seq_pred = [], []
                for p, l in zip(pred_row, label_row):
                    if l == -100:
                        continue
                    seq_true.append(id2label[l])
                    seq_pred.append(id2label[p])
                true_labels.append(seq_true)
                true_preds.append(seq_pred)
    return true_labels, true_preds


def per_entity_df(true_labels, true_preds):
    """Rapport seqeval -> DataFrame par entite + le dict brut."""
    report = classification_report(true_labels, true_preds, output_dict=True, zero_division=0)
    rows = [
        {
            "Entity": etype,
            "Precision": round(s["precision"], 4),
            "Recall": round(s["recall"], 4),
            "F1": round(s["f1-score"], 4),
            "Support": int(s["support"]),
        }
        for etype, s in report.items()
        if etype not in ("micro avg", "macro avg", "weighted avg")
    ]
    df = pd.DataFrame(rows).sort_values("Entity").reset_index(drop=True)
    return df, report

### Resultats par split (Precision / Recall / F1 / Support par entite)

In [ ]:
results = {}
for split in SPLITS:
    true_labels, true_preds = predict_split(split)
    df, report = per_entity_df(true_labels, true_preds)
    results[split] = df
    micro = report.get("micro avg", {})
    print(
        f"\n===== {split.upper()} — overall  "
        f"P: {micro.get('precision', 0):.4f}  "
        f"R: {micro.get('recall', 0):.4f}  "
        f"F1: {micro.get('f1-score', 0):.4f} ====="
    )
    display(df)

### Optionnel : sauvegarder chaque tableau en CSV

In [ ]:
for split, df in results.items():
    out_path = MODEL_DIR.parent / f"per_entity_{split}.csv"
    df.to_csv(out_path, index=False)
    print("Ecrit :", out_path)